In [37]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print(client.models.list())

SyncPage[Model](data=[Model(id='text-embedding-ada-002', created=1671217299, object='model', owned_by='openai-internal'), Model(id='whisper-1', created=1677532384, object='model', owned_by='openai-internal'), Model(id='gpt-3.5-turbo', created=1677610602, object='model', owned_by='openai'), Model(id='tts-1', created=1681940951, object='model', owned_by='openai-internal'), Model(id='gpt-3.5-turbo-16k', created=1683758102, object='model', owned_by='openai-internal'), Model(id='gpt-4-0613', created=1686588896, object='model', owned_by='openai'), Model(id='gpt-4', created=1687882411, object='model', owned_by='openai'), Model(id='davinci-002', created=1692634301, object='model', owned_by='system'), Model(id='babbage-002', created=1692634615, object='model', owned_by='system'), Model(id='gpt-3.5-turbo-instruct', created=1692901427, object='model', owned_by='system'), Model(id='gpt-3.5-turbo-instruct-0914', created=1694122472, object='model', owned_by='system'), Model(id='gpt-3.5-turbo-1106', 

In [38]:

POLICY_RULES = """
=== MOTOR INSURANCE POLICY RULES ===

1. COVERED EVENTS
The policy covers:
- Accidental collision
- Theft
- Fire
- Flood
- Third-party property damage

2. EXCLUSIONS
The policy does NOT cover:
- Driving under influence of alcohol or drugs
- Illegal racing
- Intentional damage
- Vehicle used outside permitted geographic area
- Claim filed more than 30 days after incident without valid reason

3. REQUIRED DOCUMENTS

For ALL claims:
- Claim form
- Driving license
- Vehicle registration
- Photos of damage
- Incident report

For THEFT claims only (additional):
- Police report

For THIRD-PARTY DAMAGE only (additional):
- Third-party contact information and evidence

4. ROUTING RULES
- Standard processing : covered event + complete docs + no risk flags
- Manual review       : unclear coverage / incomplete docs / conflicting info
- Fraud review        : suspicious pattern / repeated claims / weak evidence
- Rejection review    : clear exclusion applies
=== END OF POLICY RULES ===
"""

print("✅ Policy Rules loaded")
print(f"Length: {len(POLICY_RULES)} characters")

✅ Policy Rules loaded
Length: 988 characters


In [ ]:
SYSTEM_PROMPT = f"""
You are a Motor Insurance Claim Triage Assistant.

Your role is to SUPPORT human claim officers only.
You do NOT make final decisions on claims.

=== POLICY RULES ===
{POLICY_RULES}
=== END OF POLICY RULES ===

GUARDRAILS:
- If information is missing or unclear, say "Cannot determine" — do NOT guess
- If confidence is Low and NO fraud signal detected,recommend Manual review
- Never approve, reject, or calculate payment amounts
- Base your assessment ONLY on the policy rules provided above
- If ANY fraud signal is detected:
  → Set Initial Coverage Assessment to "Cannot determine"
  → Set Confidence Level to "Low"
  → Set Recommended Routing to "Fraud review"
- If ANY exclusion risk applies but cannot be fully confirmed:
  → Set Initial Coverage Assessment to "Cannot determine"
  → Set Confidence Level to "Medium"


INSTRUCTIONS:
Before checking required documents, you must:
1. Identify the claim type from the description:
   - If another vehicle or third party is involved → third-party damage
   - If vehicle is missing/stolen → theft
   - Otherwise → collision / fire / flood
2. Then check required documents based on that claim type
3. Flag any documents missing for that specific claim type
4. Check for fraud signals — flag as Fraud review if ANY of these appear:
   - Customer has submitted more than 3 claims within 12 months
   - Damage described as severe but evidence is weak or unclear
   - Inconsistent or conflicting information in the description
   - Story or timeline does not match submitted documents

OUTPUT FORMAT (follow exactly):
Claim Summary: [2-3 sentences]
Initial Coverage Assessment: [Likely covered / Possibly covered / Not covered / Cannot determine]
Missing Information: [list items or write "None"]
Risk Flags: [list items or write "None"]
Recommended Routing: [Standard processing / Manual review / Fraud review / Rejection review]
Reasoning: [explain clearly why]
Confidence Level: [High / Medium / Low]
"""

print("✅ System Prompt ready")
print(f"Length: {len(SYSTEM_PROMPT)} characters")

✅ System Prompt ready
Length: 3095 characters


In [56]:

from datetime import datetime
def analyze_claim(claim: dict) -> str:
    
    incident  = datetime.strptime(claim['incident_date'],  "%Y-%m-%d")
    submitted = datetime.strptime(claim['submitted_date'], "%Y-%m-%d")
    days_diff = (submitted - incident).days

    user_prompt = f"""
Please analyze this insurance claim:

Claim ID       : {claim['claim_id']}
Customer       : {claim['customer']}
Vehicle        : {claim['vehicle']}
Incident Date  : {claim['incident_date']}
Submitted Date : {claim['submitted_date']}
Days between incident and submission: {days_diff} days
Description    : {claim['description']}
Documents      : {claim['documents']}
Claim History  : {claim['claim_history']}
"""

    response = client.chat.completions.create(
        model       = "gpt-4o-mini",
        messages    = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt}
        ],
        temperature = 0.0,
        max_tokens  = 1000,
    )

    return response.choices[0].message.content


print("✅ Function ready")

✅ Function ready


In [ ]:

case1 = {
    "claim_id"       : "CLM-001",
    "customer"       : "Customer A",
    "vehicle"        : "Honda Civic 2022",
    "incident_date"  : "2026-05-01",
    "submitted_date" : "2026-05-05",
    "description"    : "My parked vehicle was hit by another car at a shopping mall parking lot",
    "documents"      : "Claim form, vehicle registration, photos of damage",
    "claim_history"  : "1 claim in the past 24 months"
}

result = analyze_claim(case1)
print(result)

Claim Summary: The claim involves a parked vehicle that was hit by another car in a shopping mall parking lot, indicating a third-party damage scenario. The claim was submitted within the required timeframe and includes some necessary documents, but is missing others.

Initial Coverage Assessment: Possibly covered
Missing Information: Driving license, incident report, third-party contact information and evidence
Risk Flags: None
Recommended Routing: Manual review
Reasoning: The claim is likely covered as it involves third-party damage, but it is missing several required documents, which necessitates a manual review for completeness. 
Confidence Level: Medium


In [63]:

case2 = {
    "claim_id"       : "CLM-002",
    "customer"       : "Customer B",
    "vehicle"        : "Toyota Supra 2021",
    "incident_date"  : "2026-05-03",
    "submitted_date" : "2026-05-05",
    "description"    : "The customer crashed while participating in an illegal street race",
    "documents"      : "Claim form, photos of damage, police report",
    "claim_history"  : "No prior claims"
}

result = analyze_claim(case2)
print(result)

Claim Summary: The customer reported a crash while participating in an illegal street race. The claim was submitted within 2 days of the incident, but the nature of the incident falls under an exclusion in the policy. 

Initial Coverage Assessment: Not covered
Missing Information: None
Risk Flags: [None]
Recommended Routing: Rejection review
Reasoning: The incident involves illegal racing, which is explicitly excluded from coverage under the policy rules. 
Confidence Level: High


In [64]:

case3 = {
    "claim_id"       : "CLM-003",
    "customer"       : "Customer C",
    "vehicle"        : "Toyota Camry 2020",
    "incident_date"  : "2026-05-02",
    "submitted_date" : "2026-05-06",
    "description"    : "The vehicle was stolen from a condominium parking area",
    "documents"      : "Claim form, vehicle registration",
    "claim_history"  : "No prior claims"
}

result = analyze_claim(case3)
print(result)

Claim Summary: The claim involves a theft of a Toyota Camry from a condominium parking area. The incident occurred on May 2, 2026, and the claim was submitted on May 6, 2026, which is within the 30-day submission window.

Initial Coverage Assessment: Possibly covered
Missing Information: Driving license, photos of damage, incident report, police report
Risk Flags: None
Recommended Routing: Manual review
Reasoning: The claim is for theft, which requires additional documentation such as a police report and other standard documents. Since some required documents are missing, it is recommended for manual review to ensure all necessary information is provided. 
Confidence Level: Medium


In [65]:
case4 = {
    "claim_id"       : "CLM-004",
    "customer"       : "Customer D",
    "vehicle"        : "Honda City 2020",
    "incident_date"  : "2026-05-04",
    "submitted_date" : "2026-05-06",
    "description"    : "The customer reported severe front-end damage but provided only one unclear photo",
    "documents"      : "Claim form, one unclear photo",
    "claim_history"  : "4 claims in the past 12 months"
}

result = analyze_claim(case4)
print(result)

Claim Summary: The customer reported severe front-end damage to their Honda City, but only provided one unclear photo as evidence. Additionally, the customer has submitted 4 claims in the past 12 months, which raises concerns about potential fraud.

Initial Coverage Assessment: Cannot determine
Missing Information: Vehicle registration, driving license, incident report, clear photos of damage
Risk Flags: Customer has submitted more than 3 claims within 12 months, damage described as severe but evidence is weak or unclear
Recommended Routing: Fraud review
Reasoning: The claim raises multiple risk flags, including a high number of claims in a short period and unclear evidence of damage. This necessitates a fraud review to assess the legitimacy of the claim further.
Confidence Level: Low


In [66]:
case5 = {
    "claim_id"       : "CLM-005",
    "customer"       : "Customer E",
    "vehicle"        : "Toyota Fortuner 2021",
    "incident_date"  : "2026-03-23",
    "submitted_date" : "2026-05-07",
    "description"    : "The customer reported flood damage to the vehicle",
    "documents"      : "Claim form, vehicle registration, photos of damage",
    "claim_history"  : "No prior claims"
}

result = analyze_claim(case5)
print(result)

Claim Summary: Customer E reported flood damage to their Toyota Fortuner, but the claim was submitted 45 days after the incident date. The required documents include a claim form, vehicle registration, and photos of damage, but the submission is late without a valid reason.

Initial Coverage Assessment: Cannot determine  
Missing Information: Driving license, incident report  
Risk Flags: Late submission without valid reason  
Recommended Routing: Manual review  
Reasoning: The claim was submitted more than 30 days after the incident without a valid reason, which raises a flag for possible exclusion. Additionally, the required documents are incomplete.  
Confidence Level: Medium
